# 04 - Évaluation automatique du RAG

Ce notebook illustre l'évaluation du RAG Écho à partir du jeu annoté `data/evaluation/qa_annotated.csv` et des résultats produits par `scripts/08_evaluate_rag.py`.

Il ne relance pas les appels Mistral automatiquement : il lit les fichiers de résultats s'ils existent déjà, ou affiche la commande à exécuter sinon. Cela évite de dépendre d'une clé API et de coûts d'appels pendant une simple lecture pédagogique.

In [1]:
import pandas as pd

from src.config import PATHS

QA_PATH = PATHS.data_evaluation / "qa_annotated.csv"
RESULTS_PATH = PATHS.data_evaluation / "rag_evaluation_results.csv"
SUMMARY_PATH = PATHS.data_evaluation / "rag_evaluation_summary.json"

print("Racine projet :", PATHS.root)
print("Jeu annoté    :", QA_PATH.exists())
print("Résultats     :", RESULTS_PATH.exists())
print("Résumé        :", SUMMARY_PATH.exists())

Racine projet : /Users/wld/git/oc/oc_project9_rag
Jeu annoté    : True
Résultats     : True
Résumé        : True


## 1. Jeu de test annoté

Le fichier `data/evaluation/qa_annotated.csv` contient 15 questions annotées avec leurs réponses attendues, mots-clés, identifiants d'événements attendus et commentaires d'intention.

In [2]:
qa_df = pd.read_csv(QA_PATH)
qa_df.head()

,question,expected_answer,expected_keywords,expected_event_ids,comment
0,Quels événements autour de l'astronomie sont d...,Le système doit proposer l'Initiation à l'astr...,astronomie;Lanton;observation;téléscope;nature,13708573,Requête thématique précise avec un seul événem...
1,Je cherche une exposition autour de la nature ...,Le système doit identifier l'exposition Ex(s)i...,exposition;nature;plantes;végétal sauvage;Aude...,95865579,Requête exposition / nature.
2,Y a-t-il une sortie botanique ou flore sur le ...,Le système peut proposer la sortie sur les lic...,botanique;flore;lichens;Dune du Pilat;Audenge;...,70763679;21549771,Requête thématique avec plusieurs événements a...
3,Je cherche une activité à faire en famille aut...,Le système doit proposer des événements vélo a...,vélo;famille;animations;Arès;Le Teich,80332762;38800197,Requête famille / vélo.
4,Quels événements vélo sont prévus en mai ?,Le système doit retrouver plusieurs événements...,vélo;mai;Mios;Andernos-les-Bains;La Teste-de-Buch,10772673;49861420;97562973,Requête large avec plusieurs événements simila...


In [3]:
# Statistiques simples sur le jeu de test annoté.
with_event_ids = qa_df["expected_event_ids"].fillna("").astype(str).str.strip() != ""
out_of_scope = ~with_event_ids

print(f"Nombre de questions       : {len(qa_df)}")
print(f"Colonnes                  : {list(qa_df.columns)}")
print(f"Lignes avec event_ids     : {int(with_event_ids.sum())}")
print(f"Cas hors sujet (sans IDs) : {int(out_of_scope.sum())}")

Nombre de questions       : 15
Colonnes                  : ['question', 'expected_answer', 'expected_keywords', 'expected_event_ids', 'comment']
Lignes avec event_ids     : 14
Cas hors sujet (sans IDs) : 1


In [4]:
# Visualisation simple : les colonnes utiles pour comprendre l'intention de chaque question.
qa_df[["question", "expected_keywords", "expected_event_ids", "comment"]]

,question,expected_keywords,expected_event_ids,comment
0,Quels événements autour de l'astronomie sont d...,astronomie;Lanton;observation;téléscope;nature,13708573,Requête thématique précise avec un seul événem...
1,Je cherche une exposition autour de la nature ...,exposition;nature;plantes;végétal sauvage;Aude...,95865579,Requête exposition / nature.
2,Y a-t-il une sortie botanique ou flore sur le ...,botanique;flore;lichens;Dune du Pilat;Audenge;...,70763679;21549771,Requête thématique avec plusieurs événements a...
3,Je cherche une activité à faire en famille aut...,vélo;famille;animations;Arès;Le Teich,80332762;38800197,Requête famille / vélo.
4,Quels événements vélo sont prévus en mai ?,vélo;mai;Mios;Andernos-les-Bains;La Teste-de-Buch,10772673;49861420;97562973,Requête large avec plusieurs événements simila...
5,Que peut-on faire à Arès ?,Arès;Tous en selle;spectacle;vélo;théâtre,80332762;5335717;53614677,Requête par commune.
6,Je cherche un spectacle ou une balade artistiq...,Biganos;spectacle;balade artistique;théâtre;da...,31322983,Requête spectacle / commune.
7,Y a-t-il un spectacle avec théâtre ou humour à...,Arès;théâtre;humour;musique;spectacle,5335717,Requête spectacle précise.
8,Quels événements concernent le patrimoine en s...,patrimoine;septembre;Salles;Belin-Béliet;Journ...,86073148;23079313,Requête date / patrimoine.
9,Je cherche une information santé ou accès aux ...,santé;CPAM;droits;prévention;Andernos-les-Bains,48414835,Requête santé / droits.


## 2. Lancer l'évaluation automatique

L'évaluation réelle se lance depuis la racine du projet :

```bash
poetry run python scripts/08_evaluate_rag.py
```

Ce script :

- charge le CSV annoté ;
- appelle `RagService.ask()` pour chaque question (un appel Mistral par question) ;
- calcule `keyword_match_rate`, `event_recall`, `sources_count` et un `status` (`ok` / `partial` / `ko`) ;
- écrit `data/evaluation/rag_evaluation_results.csv` (détail) et `data/evaluation/rag_evaluation_summary.json` (résumé).

Ces fichiers générés sont conservés dans le repo pour pouvoir analyser les dernières exécutions.

In [5]:
# Lecture des résultats détaillés s'ils existent déjà localement.
results_df = None
if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    display(results_df.head())
else:
    print(
        "Aucun résultat généré pour le moment. "
        "Lancez : poetry run python scripts/08_evaluate_rag.py"
    )

,question,expected_answer,generated_answer,expected_keywords,matched_keywords,keyword_match_rate,expected_event_ids,matched_event_ids,event_recall,sources_count,status,comment
0,Quels événements autour de l'astronomie sont d...,Le système doit proposer l'Initiation à l'astr...,"D'après le contexte fourni, un seul événement ...",astronomie;Lanton;observation;téléscope;nature,astronomie;Lanton;observation;nature,0.8,13708573,13708573,1.000,3,ok,Requête thématique précise avec un seul événem...
1,Je cherche une exposition autour de la nature ...,Le système doit identifier l'exposition Ex(s)i...,Voici deux événements autour de la nature et d...,exposition;nature;plantes;végétal sauvage;Aude...,exposition;nature;plantes;végétal sauvage;Aude...,1.0,95865579,95865579,1.000,3,ok,Requête exposition / nature.
2,Y a-t-il une sortie botanique ou flore sur le ...,Le système peut proposer la sortie sur les lic...,"Oui, il y a plusieurs sorties botaniques ou li...",botanique;flore;lichens;Dune du Pilat;Audenge;...,botanique;flore;lichens;Dune du Pilat;Audenge;...,1.0,70763679;21549771,70763679;21549771,1.000,3,ok,Requête thématique avec plusieurs événements a...
3,Je cherche une activité à faire en famille aut...,Le système doit proposer des événements vélo a...,Voici deux événements adaptés pour une activit...,vélo;famille;animations;Arès;Le Teich,vélo;famille;animations;Arès;Le Teich,1.0,80332762;38800197,80332762;38800197,1.000,2,ok,Requête famille / vélo.
4,Quels événements vélo sont prévus en mai ?,Le système doit retrouver plusieurs événements...,"En mai, voici les événements vélo prévus autou...",vélo;mai;Mios;Andernos-les-Bains;La Teste-de-Buch,vélo;mai;Mios,0.6,10772673;49861420;97562973,10772673,0.333,2,ok,Requête large avec plusieurs événements simila...


## 3. Statistiques sur les résultats

Si les résultats sont disponibles, on agrège les compteurs de status et les moyennes des métriques principales.

In [6]:
if results_df is not None:
    print(f"Nombre total de questions évaluées : {len(results_df)}")
    print("\nRépartition des status :")
    print(results_df["status"].value_counts())

    print(
        "\nMoyenne keyword_match_rate : "
        f"{results_df['keyword_match_rate'].mean():.3f}"
    )
    print(
        "Moyenne event_recall       : "
        f"{results_df['event_recall'].mean():.3f}"
    )
    print(
        "Moyenne sources_count      : "
        f"{results_df['sources_count'].mean():.2f}"
    )
else:
    print("results_df non disponible : statistiques ignorées.")

Nombre total de questions évaluées : 15

Répartition des status :
status
ok         14
partial     1
Name: count, dtype: int64

Moyenne keyword_match_rate : 0.850
Moyenne event_recall       : 0.811
Moyenne sources_count      : 3.67


## Analyse des cas partial / ko

Ces lignes sont celles qui méritent une revue humaine : soit la réponse n'a pas remonté les bons événements, soit elle n'a pas mentionné les mots-clés attendus, soit elle a inventé quelque chose dans un cas hors sujet.

In [7]:
if results_df is not None:
    issues_df = results_df[results_df["status"].isin(["partial", "ko"])]
    columns_to_show = [
        "question",
        "status",
        "keyword_match_rate",
        "event_recall",
        "sources_count",
        "matched_event_ids",
        "comment",
    ]
    if not issues_df.empty:
        display(issues_df[columns_to_show])
    else:
        print("Aucune ligne partial ou ko : toutes les questions sont en status ok.")
else:
    print("results_df non disponible : analyse ignorée.")

,question,status,keyword_match_rate,event_recall,sources_count,matched_event_ids,comment
14,Peux-tu me conseiller un restaurant à Arcachon ?,partial,0.333,0.0,4,NaN,Cas hors sujet : le système doit refuser d'inv...


## 5. Conclusion

- L'évaluation automatique donne une **base reproductible** : les mêmes 15 questions sont rejouées à chaque exécution, avec une graine fixe côté Mistral (`SEED=42`) pour stabiliser au mieux les réponses.
- Les métriques (`keyword_match_rate`, `event_recall`, `sources_count`) mesurent surtout **la présence d'informations clés et de sources attendues**, pas la qualité rédactionnelle de la réponse.
- L'évaluation **ne remplace pas une analyse humaine** : un `ok` ne signifie pas que la réponse est parfaite, et un `partial` peut être dû à un mot-clé absent mais une réponse pertinente.
- Les cas `partial` / `ko` servent à identifier des **pistes d'amélioration** : meilleur retrieval, filtres métier (ville / date / gratuité), reranking, ou meilleure gestion du hors sujet.